# Pipeline completo de la Fase 3 — Google Colab

Corre, en orden y sobre GPU gratuita de Colab (T4/L4):

1. **Entrenar** 4 adaptadores LoRA: Generador 1 (Groq), 2 (Cohere), 3 (Google) y la **mezcla** de los tres.
2. **Fusionar** los adaptadores de los 3 generadores con TIES, DARE+TIES y promedio lineal.
3. **Predecir**: traducir el test común (`evaluation/test_comun.json`) con 8 modelos (base sin ajustar, 4 adaptadores, 3 fusiones).
4. **Medir** BLEU y chrF de cada uno y armar la tabla comparativa.
5. **Descargar** un `.zip` con todo para traerlo al repo local.

**Antes de empezar**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**. Las celdas se corren en orden; si Colab se desconecta, vuelve a correr desde la celda 1 (las partes ya hechas se saltan solas si sus carpetas existen).

Tiempo estimado en T4: ~15-30 min por adaptador (el Generador 1 con 189 ejemplos tardó ~9 min; ahora son ~475), ~5 min por fusión, ~5 min por modelo en predicciones: **2 a 3 horas** en total.

In [ ]:
# 1. Confirmar que hay GPU asignada
!nvidia-smi

In [ ]:
# 2. Clonar el repo (público) con rutas absolutas y solo si hace falta;
# si ya existe, traer lo último de la rama develop.
import os
%cd /content
if not os.path.isdir("/content/traductor-jerga-dialectal-slm"):
    !git clone -b develop https://github.com/Anderfg13/traductor-jerga-dialectal-slm.git
%cd /content/traductor-jerga-dialectal-slm
!git pull -q origin develop

In [ ]:
# 3. Dependencias puntuales (NO requirements.txt completo: Colab ya trae
# torch con CUDA). Se desinstala torchao porque la versión vieja
# preinstalada rompe peft (ver CONTEXTO_PROYECTO.md, "CÓMPUTO PESADO").
!pip install -q transformers peft accelerate python-dotenv sacrebleu
!pip uninstall -y -q torchao

## Parte A — Datos

In [ ]:
# 4. Armar el dataset de la mezcla (1+2+3) y el test común de evaluación.
!python generation/mezclar_datasets.py
!python evaluation/construir_test_comun.py

## Parte B — Entrenar los 4 adaptadores

In [ ]:
# Entrenar: Generador 1 (Groq). Entrenamiento completo con validación y
# selección del mejor checkpoint (EarlyStopping). Se salta si ya existe.
import os
if os.path.isdir("finetuning/checkpoints/generador1/adapter"):
    print("Ya existe finetuning/checkpoints/generador1/adapter, se salta.")
else:
    !python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_generador1 --todos

In [ ]:
# Entrenar: Generador 2 (Cohere). Entrenamiento completo con validación y
# selección del mejor checkpoint (EarlyStopping). Se salta si ya existe.
import os
if os.path.isdir("finetuning/checkpoints/generador2/adapter"):
    print("Ya existe finetuning/checkpoints/generador2/adapter, se salta.")
else:
    !python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_generador2 --todos

In [ ]:
# Entrenar: Generador 3 (Google). Entrenamiento completo con validación y
# selección del mejor checkpoint (EarlyStopping). Se salta si ya existe.
import os
if os.path.isdir("finetuning/checkpoints/generador3/adapter"):
    print("Ya existe finetuning/checkpoints/generador3/adapter, se salta.")
else:
    !python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_generador3 --todos

In [ ]:
# Entrenar: Mezcla 1+2+3. Entrenamiento completo con validación y
# selección del mejor checkpoint (EarlyStopping). Se salta si ya existe.
import os
if os.path.isdir("finetuning/checkpoints/mezcla/adapter"):
    print("Ya existe finetuning/checkpoints/mezcla/adapter, se salta.")
else:
    !python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_mezcla --todos

In [ ]:
# Curvas de pérdida de los 4 entrenamientos (para el paper y para revisar sobreajuste)
import json
for c in ["generador1", "generador2", "generador3", "mezcla"]:
    print("=====", c)
    print(open(f"finetuning/checkpoints/{c}/loss_log.json").read()[:1500])

In [ ]:
# Verificar que los 4 adaptadores salieron con la MISMA configuración de LoRA
# (condición para que la comparación de PI1 sea válida). Debe terminar en OK.
!python finetuning/verificar_config_identica.py

## Parte C — Fusión de adaptadores (PI2)

TIES, DARE+TIES y promedio lineal de los adaptadores de los generadores 1, 2 y 3. `--densidad 0.5`: se conserva la mitad de los parámetros de cada adaptador (valor estándar; se puede variar).

In [ ]:
G = "finetuning/checkpoints/generador{}/adapter"
ADAPTADORES = f"{G.format(1)} {G.format(2)} {G.format(3)}"
for metodo in ["ties", "dare_ties", "linear"]:
    salida = f"finetuning/checkpoints/fusion_{metodo}"
    if os.path.isdir(salida + "/fusion"):
        print(f"Ya existe {salida}/fusion, se salta.")
    else:
        !python merging/fusionar_adaptadores.py --metodo {metodo} --adaptadores {ADAPTADORES} --salida {salida}

### Fusión guiada por destilación multi-maestro

Un adaptador estudiante (arranca desde la fusión TIES) aprende a imitar el promedio de los 3 adaptadores-maestro sobre el dataset mezcla. Ver `merging/destilacion_multimaestro.py`. Tarda ~20-40 min en T4.

In [ ]:
if os.path.isdir("finetuning/checkpoints/destilacion/estudiante"):
    print("Ya existe la destilación, se salta.")
else:
    !python merging/destilacion_multimaestro.py --maestros finetuning/checkpoints/generador1/adapter finetuning/checkpoints/generador2/adapter finetuning/checkpoints/generador3/adapter --inicial finetuning/checkpoints/fusion_ties/fusion --dataset-dir generation/splits/dataset_mezcla --salida finetuning/checkpoints/destilacion

## Parte D — Predicciones sobre el test común

In [ ]:
MODELOS = {
    "baseline": None,
    "generador1": "finetuning/checkpoints/generador1/adapter",
    "generador2": "finetuning/checkpoints/generador2/adapter",
    "generador3": "finetuning/checkpoints/generador3/adapter",
    "mezcla": "finetuning/checkpoints/mezcla/adapter",
    "fusion_ties": "finetuning/checkpoints/fusion_ties/fusion",
    "fusion_dare_ties": "finetuning/checkpoints/fusion_dare_ties/fusion",
    "fusion_linear": "finetuning/checkpoints/fusion_linear/fusion",
    "destilacion": "finetuning/checkpoints/destilacion/estudiante",
}
for nombre, adapter in MODELOS.items():
    if os.path.exists(f"evaluation/predicciones/{nombre}.json"):
        print(f"Ya existen las predicciones de {nombre}, se salta.")
        continue
    flag = f"--adapter {adapter}" if adapter else ""
    !python evaluation/generar_predicciones.py --nombre {nombre} {flag}

## Parte E — Métricas y tabla comparativa

In [ ]:
os.makedirs("evaluation/reportes", exist_ok=True)
for nombre in MODELOS:
    print("=====", nombre)
    !python evaluation/metricas_automaticas.py --predicciones evaluation/predicciones/{nombre}.json --referencias evaluation/test_comun.json --campo-prediccion traduccion_modelo --salida evaluation/reportes/{nombre}.md --salida-json evaluation/reportes/{nombre}.json | head -12
!python evaluation/tabla_comparativa.py

## Parte F — Empaquetar y descargar

In [ ]:
# Solo lo necesario: adaptadores finales (NO los checkpoints intermedios del
# Trainer, que pesan mucho), pérdidas, predicciones y reportes.
!rm -f resultados_fase3.zip
!zip -rq resultados_fase3.zip finetuning/checkpoints/generador1/adapter finetuning/checkpoints/generador1/loss_log.json finetuning/checkpoints/generador2/adapter finetuning/checkpoints/generador2/loss_log.json finetuning/checkpoints/generador3/adapter finetuning/checkpoints/generador3/loss_log.json finetuning/checkpoints/mezcla/adapter finetuning/checkpoints/mezcla/loss_log.json finetuning/checkpoints/fusion_ties/fusion finetuning/checkpoints/fusion_dare_ties/fusion finetuning/checkpoints/fusion_linear/fusion finetuning/checkpoints/destilacion/estudiante evaluation/predicciones evaluation/reportes evaluation/comparacion_fase3.md
!ls -lh resultados_fase3.zip
from google.colab import files
files.download("resultados_fase3.zip")